# Day 22: LangChain RAG Chains & Automated Retrieval

Welcome to Day 22 of the AI Engineering Mastery program!

Today, we are migrating from manual vector database queries to using **LangChain's specialized RAG chains**. This shift allows us to elegantly compose retrieval mechanisms with LLM generation, cleanly separating concerns and drastically reducing boilerplate code.

## Core Theory: The "Why" and "How"

### Why move away from manual retrieval?
Previously, you might have written code that explicitly queries Qdrant for vectors, formats those documents into a string, and injects them into an LLM prompt. In a production environment, this approach becomes fragile:
- **Scalability**: Adding memory, chat history, or fallback retrievers introduces complex branching.
- **Formatting**: Handling different document formats and metadata becomes tedious.
- **Observability**: Manual orchestration makes it harder to trace the lifecycle of a prompt (e.g., using LangSmith).

### How LangChain Chains Work
LangChain provides `create_retrieval_chain` and document combining chains (like `create_stuff_documents_chain`).
- **Document Chain**: Takes retrieved documents and "stuffs" them into the LLM prompt. 
- **Retrieval Chain**: Wraps the Document Chain. It takes the user's input, fetches relevant documents from the retriever, and passes them to the Document Chain.

This decouples the *fetching* of information from the *reasoning* over that information.

## Common Pitfalls in Production

1. **Missing Prompt Variables**: `create_stuff_documents_chain` requires a prompt with a `context` input variable. Omitting this or naming it differently will cause runtime validation errors.
2. **Over-Stuffing the Context Window**: The "stuff" chain dumps all retrieved documents into the prompt. If your retriever returns too many chunks, or very large chunks, you will easily hit the LLM's token limit.
3. **Synchronous Bottlenecks**: Retrieving and generating are IO-bound tasks. In high-concurrency production systems, use the asynchronous interfaces (`ainvoke`, `abatch`) to prevent blocking your application loop.
4. **Ignoring MMR (Maximal Marginal Relevance)**: Relying solely on similarity search can retrieve redundant documents. Using MMR balances relevance with diversity, ensuring the LLM sees broader context rather than repeated phrasing.

## Code Implementation & Practical Lab

**Task**: Build a fully operational RAG chain using Qdrant and LangChain. We will use OpenAI Embeddings and Chat models to construct a production-ready application.

In [1]:
import sys
!{sys.executable} -m pip install -qU langchain langchain-core langchain-qdrant qdrant-client langchain-openai


/app/.venv/bin/python3: No module named pip


### Tier 1: Basic (Core Concept)
This example isolates the bare minimum required to construct a retrieval chain using LangChain. It focuses purely on composing the chain without external vector store setup overhead, handling fallback directly if it fails.

In [2]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import os

# 1. Setup store and embeddings (uses try/except to avoid crash if no API key)
try:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings)
    vectorstore.add_documents([Document(page_content="LangChain retrieval chains decouple retrieval from generation.")])
    retriever = vectorstore.as_retriever()
    
    # 2. Setup prompt and LLM
    llm = ChatOpenAI(model="gpt-4o-mini")
    prompt = ChatPromptTemplate.from_template(
        "Answer based on context:\n{context}\n\nQuestion: {input}"
    )
    
    # 3. Compose chains
    document_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, document_chain)
    print("Basic RAG chain instantiated successfully!")
except Exception as e:
    print(f"Basic RAG setup skipped (API failure or missing key): {e}")


Basic RAG setup skipped (API failure or missing key): Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


### Tier 2: Medium (OOP & State Management)
Here we encapsulate the logic within a class, cleanly managing the lifecycle of the Qdrant client, the embedding model, and the LLM. This structure allows the RAG service to be easily integrated into a larger application.

In [3]:
from typing import List
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

class MediumRAGService:
    def __init__(self, collection_name: str = "medium_rag"):
        # Initialize external services
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.llm = ChatOpenAI(model="gpt-4o-mini")
        
        # Setup Qdrant
        self.client = QdrantClient(":memory:")
        self.client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
        )
        self.vectorstore = QdrantVectorStore(
            client=self.client,
            collection_name=collection_name,
            embedding=self.embeddings,
        )
        self.retriever = self.vectorstore.as_retriever()
        
        # Compose Chain
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Answer using context:\n{context}"),
            ("human", "{input}"),
        ])
        doc_chain = create_stuff_documents_chain(self.llm, prompt)
        self.rag_chain = create_retrieval_chain(self.retriever, doc_chain)
        
    def add_texts(self, texts: List[str]) -> None:
        docs = [Document(page_content=t) for t in texts]
        self.vectorstore.add_documents(docs)
        
    def query(self, user_query: str) -> dict:
        # Returns a dict with 'answer' and 'context'
        return self.rag_chain.invoke({"input": user_query})

print("Medium RAGService class defined successfully!")


Medium RAGService class defined successfully!


### Tier 3: Advanced (Production-Grade & AI Security)
This builds on the original RAG implementation by adding production-level rigor:
- **Graceful Fallbacks**: Wrapping the invocation in a `try/except` block to prevent application crashes.
- **PII Redaction**: Intercepting and sanitizing user inputs before they are passed to the retriever or LLM.

In [4]:
import os
import re
import logging
from typing import List, Any

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def setup_qdrant_vectorstore(collection_name: str = "day_22_collection") -> QdrantVectorStore:
    """
    Initializes an in-memory Qdrant client and a LangChain VectorStore.
    
    Args:
        collection_name (str): The name of the Qdrant collection.
        
    Returns:
        QdrantVectorStore: A configured LangChain vector store.
    """
    client = QdrantClient(":memory:")
    
    # OpenAI's text-embedding-3-small uses 1536 dimensions
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )
    
    return vectorstore

def populate_vectorstore(vectorstore: QdrantVectorStore, docs: List[Document]) -> None:
    """
    Populates the vector store with initial documents.
    
    Args:
        vectorstore (QdrantVectorStore): The target vector store.
        docs (List[Document]): The documents to ingest.
    """
    vectorstore.add_documents(docs)

def build_rag_chain(vectorstore: QdrantVectorStore, llm: Any) -> Any:
    """
    Constructs the end-to-end RAG chain using LangChain's built-in retrieval chain factories.
    
    Args:
        vectorstore (QdrantVectorStore): The vector store to use for retrieval.
        llm (Any): The LLM to use for generation.
        
    Returns:
        Any: A LangChain executable chain.
    """
    # 1. Configure the retriever. We use MMR (Maximal Marginal Relevance) 
    #    to fetch diverse documents, guarding against redundant context.
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    )
    
    # 2. Define the prompt template. It MUST include a {context} variable 
    #    for the stuffed documents, and an {input} variable for the user query.
    system_prompt = (
        "You are a highly capable AI assistant specializing in software engineering.\n"
        "Use the following pieces of retrieved context to answer the user's question.\n"
        "If you do not know the answer, simply state that you don't know.\n\n"
        "{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    # 3. Create the document combining chain
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    
    # 4. Create the final retrieval chain
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    
    return rag_chain

def redact_pii(text: str) -> str:
    """
    AI Security Layer: Redacts simple PII (like emails) from input strings.
    """
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    return re.sub(email_pattern, '[REDACTED_EMAIL]', text)

def safe_query_chain(chain, user_query: str) -> dict:
    """
    Executes a query with PII redaction and a fallback mechanism on failure.
    """
    safe_query = redact_pii(user_query)
    try:
        return chain.invoke({"input": safe_query})
    except Exception as e:
        logger.error(f"RAG chain invocation failed: {e}")
        return {
            "input": safe_query,
            "answer": "I am currently experiencing technical difficulties. Please try again later.",
            "context": []
        }

# ==========================================
# Lab Execution
# ==========================================
if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = "sk-dummy"
        
    try:
        print("Initializing Qdrant VectorStore...")
        vs = setup_qdrant_vectorstore()
        
        sample_docs = [
            Document(page_content="LangChain's create_retrieval_chain wraps a retriever and a document chain.", metadata={"source": "docs"}),
            Document(page_content="Qdrant is a fast, scalable vector search engine.", metadata={"source": "docs"}),
            Document(page_content="MMR balances relevance and diversity in search results.", metadata={"source": "docs"}),
            Document(page_content="Always type-hint your Python code for production safety.", metadata={"source": "best_practices"})
        ]
        
        print("Ingesting documents...")
        populate_vectorstore(vs, sample_docs)
        
        print("Initializing LLM...")
        # Adding a timeout for production readiness
        llm = ChatOpenAI(model="gpt-4o-mini", request_timeout=10.0)
        
        print("Building RAG Chain...")
        chain = build_rag_chain(vs, llm)
        
        query = "What is the role of admin@example.com in LangChain retrieval?"
        print(f"\nExecuting Query: {query}")
        
        response = safe_query_chain(chain, query)
        print("\n--- Response ---")
        print(response.get("answer"))
        print("\n--- Retrieved Context Sources ---")
        for doc in response.get("context", []):
            print(f"- {doc.page_content}")
    except Exception as e:
        print(f"\nLab setup aborted due to missing valid API key or network error: {e}")


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 401 Unauthorized"


Initializing Qdrant VectorStore...

Lab setup aborted due to missing valid API key or network error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


## Practical Lab / Homework

**Task**: Refactor the Advanced Tier implementation to integrate a "memory" component. Currently, our chains answer single queries without context of past turns. 

1. Modify the prompt to accept `chat_history`.
2. Wrap the chain in a `create_history_aware_retriever` so the retriever can reformulate questions based on history.
3. (Optional but recommended) Record a 3-minute async video walkthrough (e.g., using Loom) explaining your design decisions regarding state management and how you maintained clean OOP boundaries.


## Reference Links

- [LangChain Retrieval Chains Documentation](https://python.langchain.com/docs/modules/data_connection/retrievers/)
- [Qdrant LangChain Integration](https://qdrant.tech/documentation/frameworks/langchain/)
- [OWASP Top 10 for LLM Applications (For PII & Security considerations)](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
